In [4]:
import pandas as pd
import glob

## Função para processar arquivos com as colunas que vamos usar


In [ ]:

def processar_arquivo(input_path, output_path):

    print(f"Processando: {input_path}")

    df = pd.read_parquet(input_path)

    df = df[[
    1,
    7,
    30,
    31,
    32,
    33,
    34
    ]]

    df.columns = [
        "date",
        "actor1_country",
        "goldstein",
        "mentions",
        "sources",
        "articles",
        "tone"
    ]

    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d", errors="coerce")
    df["goldstein"] = pd.to_numeric(df["goldstein"], errors="coerce")
    df["tone"] = pd.to_numeric(df["tone"], errors="coerce")
    df["mentions"] = pd.to_numeric(df["mentions"], errors="coerce")
    df["sources"] = pd.to_numeric(df["sources"], errors="coerce")
    df["articles"] = pd.to_numeric(df["articles"], errors="coerce")

    df = df.dropna(subset=["actor1_country", "date"])

    df["tone"] = df["tone"].fillna(0)
    df["goldstein"] = df["goldstein"].fillna(0)

    # salvar
    df.to_parquet(output_path)

    print(f"Salvo: {output_path}")


## Carregando e juntando os parquets em um so DF

In [1]:
def carregar_dados(path_pattern):

    files = glob.glob(path_pattern)

    df = pd.concat(
        [pd.read_parquet(f) for f in files],
        ignore_index=True
    )

    return df

## Funçao para calcular a Feature World Tension Index

In [2]:
def calcular_wti(df):

    df = df.copy()

    # normalizar goldstein (-10 a 10 → 0 a 1 invertido)
    df['goldstein_norm'] = (10 - df['goldstein']) / 20

    # tone negativo normalizado
    df['tone_neg'] = df['tone'].apply(lambda x: abs(x) if x < 0 else 0)
    df['tone_norm'] = df['tone_neg'] / 100

    #Criando uma feature de Midia, impacto da midia no WTI
    df['media_weight'] = (
        df['mentions'] * 0.5 +
        df['articles'] * 0.3 +
        df['sources'] * 0.2
    )
    df['media_weight'] = df['media_weight'] / df['media_weight'].max()
    
    # WTI
    df['wti_raw'] = (
        df['goldstein_norm'] * 0.6 +
        df['tone_norm'] * 0.2 +
        df['media_weight'] * 0.2
    )
        
    #Normalizando
    df['wti'] = (
        (df['wti_raw'] - df['wti_raw'].min()) /
        (df['wti_raw'].max() - df['wti_raw'].min())
    )
    return df



In [5]:
import os

RAW_DIR = "data/raw/historical"
PROCESSED_DIR = "data/processed/historical"

os.makedirs(PROCESSED_DIR, exist_ok=True)


files = glob.glob(f"{RAW_DIR}/*.parquet")

for file in files:
    
    filename = os.path.basename(file)
    
    output_path = os.path.join(PROCESSED_DIR, filename)
    
    processar_arquivo(file, output_path)

NameError: name 'processar_arquivo' is not defined

In [ ]:
import os

RAW_DIR = "data/raw/live"
PROCESSED_DIR = "data/processed/live"


os.makedirs(PROCESSED_DIR, exist_ok=True)


files = glob.glob(f"{RAW_DIR}/*.parquet")

for file in files:
    
    filename = os.path.basename(file)
    
    output_path = os.path.join(PROCESSED_DIR, filename)

    processar_arquivo(file, output_path)

In [14]:
df = carregar_dados("data/processed/historical/*.parquet")
df = df.groupby(["date", "actor1_country"]).agg({
    "goldstein": "mean",
    "tone": "mean",
    "mentions": "sum",
    "sources": "sum",
    "articles": "sum"
}).reset_index()
df = df.sort_values(["actor1_country", "date"])
df = calcular_wti(df)
df = df.set_index("actor1_country")

In [24]:
df.loc[input("Enter country: ").upper()].head(10)

,date,goldstein,tone,mentions,sources,articles,goldstein_norm,tone_neg,tone_norm,media_weight,wti_raw,wti
actor1_country,,,,,,,,,,,,
ISR,2015-01-04,1.900000,-5.717514,2,1,2,0.405000,5.717514,0.057175,0.000027,0.254440,0.407722
ISR,2015-01-07,1.900000,-6.376161,2,1,2,0.405000,6.376161,0.063762,0.000027,0.255758,0.409841
ISR,2024-01-02,-0.984615,-5.018533,47,13,47,0.549231,5.018533,0.050185,0.000609,0.339697,0.544862
ISR,2024-01-03,-6.606667,-5.715215,153,48,153,0.830333,5.715215,0.057152,0.002000,0.510030,0.818851
ISR,2024-01-04,-5.270968,-6.462363,103,31,103,0.763548,6.462363,0.064624,0.001343,0.471322,0.756587
ISR,2024-01-05,-3.966667,-3.634740,40,14,40,0.698333,3.634740,0.036347,0.000527,0.426375,0.684287
ISR,2024-01-06,-3.662857,-5.801348,110,36,110,0.683143,5.801348,0.058013,0.001443,0.421777,0.676891
ISR,2024-01-07,-6.135714,-4.556838,62,14,62,0.806786,4.556838,0.045568,0.000794,0.493344,0.792010
ISR,2024-12-02,-3.200000,-4.360860,9,3,9,0.660000,4.360860,0.043609,0.000118,0.404745,0.649495
